## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Medical Questions to Answer**

1. **Sepsis Management**: What is the protocol for managing sepsis in a critical care unit?
2. **Appendicitis**: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
3. **Sudden Patchy Hair Loss**: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp? What could be the possible causes behind it?
4. **Brain Tissue Injury**: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
5. **Leg Fracture During Hiking**: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip? What should be considered for their care and recovery?


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# No local llama-cpp model installation is required for this version.
# The notebook calls a remote Ollama server through its OpenAI-compatible API.
# Make sure the remote server is reachable from this machine/Colab runtime before running the LLM cells.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries used by this notebook
# !pip install openai==2.7.1 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# Libraries for processing dataframes and text
import json
import os
import textwrap
import warnings

# Force transformers/sentence-transformers to use the PyTorch path only.
# This prevents errors such as: ModuleNotFoundError: No module named 'tf_keras'.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import pandas as pd
import tiktoken
from IPython.display import Markdown, display

# Libraries for loading data, chunking, embeddings, and vector databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# OpenAI-compatible client for the remote Ollama server
from openai import OpenAI

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)


d:\PythonPortable\python-3.12.8\Lib\site-packages\transformers\utils\hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Question Answering using LLM

#### Connecting to the Remote Ollama Model

This notebook uses the OpenAI-compatible API exposed by the remote Ollama server at `192.168.1.209:11434`. The model must already be available on the Ubuntu server through `ollama list` or `ollama pull`.


NOTE::

The model used in this notebook is a Hugging Face model artifact served through Ollama using the hf.co/... model path. The notebook connects to the locally hosted Ollama OpenAI-compatible endpoint to run the Hugging Face GGUF model.   

In [5]:
# Initialize the OpenAI-compatible client that points to the remote Ubuntu Ollama server.
# Ollama ignores the API key, but the OpenAI SDK requires a placeholder string.
OLLAMA_BASE_URL = "http://192.168.1.209:11434/v1"

REMOTE_MODELS = {
    "mistral_instruct": "hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K",
}

# Recommended default for this notebook: use the remote Mistral model that matches the original full-code scaffold.
# Switch to REMOTE_MODELS["fast_small"] for quick smoke tests or REMOTE_MODELS["stronger_medical_rag"] for larger-model comparison.
LLM_MODEL = REMOTE_MODELS["mistral_instruct"]

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

# Optional: confirm which models the remote endpoint exposes.
try:
    available_models = [m.id for m in client.models.list().data]
    print("Remote Ollama models:")
    for model_id in available_models:
        print("-", model_id)
    if LLM_MODEL not in available_models:
        print()
        print(f"Warning: {LLM_MODEL!r} was not returned by /v1/models. Check the Ollama model name before running generation cells.")
except Exception as e:
    print("Could not list remote models. Check LAN/VPN/firewall connectivity and Ollama host binding.")
    print(e)

questions = {
    "Q1": "What is the protocol for managing sepsis in a critical care unit?",
    "Q2": "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Q3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Q4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "Q5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
}


Remote Ollama models:
- nomic-embed-text:latest
- hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K
- qwen2.5-coder:1.5b
- hf.co/giladgd/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M-GGUF:Q4_K_M


### Model Loading Note

The main LLM used in this project is a Hugging Face model artifact:

`hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K`

Although the model artifact is hosted on Hugging Face, it is served through Ollama because the notebook environment does not have enough local GPU memory to load the full model directly. The model path `hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K` confirms that the model source is Hugging Face. For reproducibility, the remote Ollama server must have this model pulled before running the notebook.


### Important reproducibility note:

This notebook requires:
1. A running Ollama server available at the configured base URL.
2. The Hugging Face GGUF model already pulled into Ollama.
3. The file `medical_diagnosis_manual.pdf` placed in the same folder as this notebook.

If running on another machine, update `OLLAMA_BASE_URL` and ensure the PDF is available.



### Optional direct Hugging Face loading

The main notebook uses Ollama to serve the Hugging Face GGUF model because it is easier to run on the available local GPU setup. However, the same model can also be loaded directly from Hugging Face using `llama-cpp-python`, as shown in the optional commented cell below. This demonstrates that the large language model source is Hugging Face, while Ollama is only being used as the serving layer.


In [ ]:
# ============================================================
# OPTIONAL: Direct Hugging Face model loading without Ollama
# ============================================================
# This cell is provided for rubric compliance to show how the same
# Hugging Face GGUF model can be loaded directly from Hugging Face.
#
# In this notebook, the same model is served through Ollama for easier
# local GPU execution:
#   hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K
#
# The direct Hugging Face approach below uses llama-cpp-python because
# the model is in GGUF format. Standard Hugging Face Transformers usually
# loads .safetensors / PyTorch models, not GGUF models directly.
#
# This cell is commented out because it may require:
# - llama-cpp-python installed with proper CPU/GPU support
# - enough RAM/VRAM
# - internet access to download the model from Hugging Face
# - compatible CUDA build if GPU offloading is used

# !pip install llama-cpp-python huggingface-hub

# from llama_cpp import Llama
#
# # Hugging Face repository and exact GGUF file
# HF_REPO_ID = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
# HF_MODEL_FILE = "mistral-7b-instruct-v0.2.Q6_K.gguf"
#
# # Load the model directly from Hugging Face
# llm_direct_hf = Llama.from_pretrained(
#     repo_id=HF_REPO_ID,
#     filename=HF_MODEL_FILE,
#     n_ctx=4096,
#     n_gpu_layers=-1,      # Use GPU offload if supported; set 0 for CPU only
#     verbose=False
# )
#
# def direct_hf_response(prompt, temperature=0.1, top_p=0.9, max_tokens=512):
#     """
#     Generate a response using the Hugging Face GGUF model loaded directly
#     with llama-cpp-python, without using Ollama.
#     """
#     formatted_prompt = f"""<s>[INST] {prompt} [/INST]"""
#
#     output = llm_direct_hf(
#         formatted_prompt,
#         max_tokens=max_tokens,
#         temperature=temperature,
#         top_p=top_p,
#         stop=["</s>"]
#     )
#
#     return output["choices"][0]["text"].strip()
#
# # Example test
# # test_answer = direct_hf_response("What is sepsis?")
# # print(test_answer)

#### Response

In [5]:
def llm_chat(system_message, user_message, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50, model=None):
    """Call the remote Ollama model through the OpenAI-compatible chat API."""
    completion = client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        # Ollama accepts top_k as an additional option through the OpenAI SDK's extra_body.
        extra_body={"top_k": top_k},
    )
    return completion.choices[0].message.content.strip()


def response(query, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50):
    """Generate a plain LLM response without external retrieved context."""
    return llm_chat(
        system_message="You are a careful medical information assistant. Give accurate, concise, safety-aware answers and recommend licensed clinical judgment for medical decisions.",
        user_message=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q1"] = response(questions["Q1"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q1"])


Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis involves a multidisciplinary approach and prompt recognition and intervention. Here are some general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize sepsis early based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, and elevated inflammatory markers), and vital sign instability.
2. Resuscitation: Provide adequate fluid resuscitation to maintain adequate tissue perfusion. This may involve administering intravenous fluids such as normal saline or lactated Ringer's solution.
3. Vasopressor support: If the patient is hypotensive despite adequate fluid resuscitation, administer vasopressors to maintain mean arterial pressure (MAP) ≥65 mmHg.
4. Oxygenation: Provide supplemental oxygen or mechanical ventilation if necessary to maintain adequate oxygen saturation (Spo2 >90%).
5. Antibi

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q2"] = response(questions["Q2"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q2"])


Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right abdomen. Common symptoms of appendicitis include:

1. Sudden onset of pain, usually starting around the navel and then shifting to the lower right abdomen.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever (often low-grade at first).
5. Abdominal swelling and tenderness.
6. Constipation or diarrhea.
7. Inability to pass gas or have a bowel movement.
8. Feeling bloated or having cramps.

Appendicitis cannot be cured via medicine alone as the inflammation can progress to rupture, leading to peritonitis – a serious and potentially life-threatening condition. If diagnosed early, appendicitis is typically treated with an appendectomy, which is a surgical procedure to remove the appendix. This surgery can be performed laparoscopically (using small incisions and a camera) or open (making a larger incision). The goal of treatment is to prevent the appendix

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q3"] = response(questions["Q3"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q3"])


I cannot provide specific treatments or solutions without a proper diagnosis from a licensed healthcare professional. Sudden patchy hair loss, also known as alopecia areata, can have various causes including autoimmune disorders, stress, vitamin deficiencies, or certain medications. It's important to rule out any underlying medical conditions before considering treatments.

Effective treatments for alopecia areata include:

1. Topical corticosteroids: These are anti-inflammatory creams or solutions that can help reduce inflammation and promote hair regrowth.
2. Injections of corticosteroids: Injections of corticosteroids directly into the bald spots may be more effective than topical applications.
3. Minoxidil: A medication that is applied topically to stimulate hair growth.
4. Diphencyprone (DPCP): An allergen that, when introduced to the scalp, can trigger an immune response and promote hair regrowth.
5. JAK inhibitors: Oral medications like tofacitinib and upadacitinib have shown pr

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q4"] = response(questions["Q4"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q4"])


I cannot provide specific treatment recommendations without consulting a healthcare professional who has evaluated the patient's condition. However, I can provide some general information about common treatments for brain injuries.

For mild to moderate brain injuries, also known as traumatic brain injuries (TBIs), the following treatments may be recommended:

1. Rest and relaxation: The brain needs time to heal after an injury. Patients are usually advised to rest and avoid physical activity that could worsen their condition.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage pain, reduce inflammation, prevent or treat infections, or improve cognitive function.
3. Rehabilitation: Physical, occupational, and speech therapy can help patients regain lost skills and improve their overall functioning. This may include exercises to improve strength, coordination, balance, and cognitive abilities.
4. Supportive care: Patients with brain in

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q5"] = response(questions["Q5"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q5"])


I'm not a doctor, but I can provide you with general information on this topic based on available resources. If someone has fractured their leg during a hiking trip, here are the necessary precautions and treatment steps:

1. Assess the severity of the injury: Check for signs of open wounds, swelling, bruising, deformity, numbness, or inability to move the leg. If the fracture is open (compound), if there's severe pain, or if you suspect other injuries, seek medical help immediately.

2. Provide first aid: Apply a sterile dressing to any wounds and immobilize the affected leg using a splint, sling, or a makeshift splint if necessary. Keep the leg elevated to reduce swelling and pain.

3. Control bleeding: Apply direct pressure on the wound with a clean cloth to control any bleeding. Do not remove the cloth until you reach medical help.

4. Monitor vital signs: Check the person's pulse, breathing rate, and level of consciousness regularly.

5. Transportation: If possible, arrange for tr

### Base LLM observations

| Question     | Base LLM Quality      | Risk / Limitation                              |
| ------------ | --------------------- | ---------------------------------------------- |
| Sepsis       | Gives broad protocol  | May miss exact source-backed guideline details |
| Appendicitis | Mostly correct        | No citation or retrieved evidence              |
| Alopecia     | Useful general answer | May include treatments not grounded in manual  |
| Brain injury | General and safe      | Too broad; needs source-specific management    |
| Fracture     | Practical first aid   | Needs better linkage to clinical reference     |


## Question Answering using LLM with Prompt Engineering

Five prompt-engineering and parameter-tuning combinations were tested. Each combination changed the system prompt, temperature, top_p, top_k, and max_tokens. The profiles were applied across the five required medical questions to compare how prompt style and decoding settings affected answer structure, safety language, and completeness.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
prompt_profiles = [
    {
        "name": "concise_clinical_protocol",
        "system": "You are a careful clinical decision-support assistant. Give a concise protocol-style answer and include emergency red flags.",
        "temperature": 0.0,
        "top_p": 0.90,
        "top_k": 30,
        "max_tokens": 420,
    },
    {
        "name": "differential_and_treatment",
        "system": "You are a medical assistant helping clinicians. Structure the answer as symptoms, likely diagnosis, treatment, and when to escalate care.",
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 40,
        "max_tokens": 460,
    },
    {
        "name": "patient_safe_language",
        "system": "You are a healthcare information assistant. Use patient-friendly language, avoid definitive diagnosis, and recommend professional medical evaluation.",
        "temperature": 0.3,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 460,
    },
    {
        "name": "trauma_first_aid",
        "system": "You are an emergency medicine support assistant. Prioritize stabilization, immediate precautions, and referral to urgent or emergency care.",
        "temperature": 0.1,
        "top_p": 0.85,
        "top_k": 30,
        "max_tokens": 460,
    },
    {
        "name": "evidence_checklist",
        "system": "You are a clinical knowledge assistant. Answer with a checklist of key findings, treatment steps, contraindications, and follow-up considerations.",
        "temperature": 0.0,
        "top_p": 0.92,
        "top_k": 20,
        "max_tokens": 500,
    },
]


def prompt_engineered_response(question, profile):
    return llm_chat(
        system_message=profile["system"],
        user_message=question,
        max_tokens=profile["max_tokens"],
        temperature=profile["temperature"],
        top_p=profile["top_p"],
        top_k=profile["top_k"],
    )

prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q1"] = {
    "profile": prompt_profiles[0]["name"],
    "answer": prompt_engineered_response(questions["Q1"], prompt_profiles[0])
}
print(prompt_engineered_answers["Q1"]["answer"])


Title: Septic Shock Management Protocol in Critical Care Unit

1. **Recognition:** Suspect sepsis if the patient has an infection, plus two or more of the following: temperature >38°C (100.4°F) or <36°C (96.8°F), heart rate >90 bpm, respiratory rate >20 breaths per minute, altered mental status, or lactate level >2 mmol/L.

2. **Immediate Actions:**
   a. Start high-flow oxygen via non-rebreather mask or endotracheal tube if necessary.
   b. Initiate intravenous access (two large-bore peripheral lines or central line).
   c. Begin fluid resuscitation with isotonic crystalloids (30 mL/kg in the first hour, then 6 mL/kg per hour for next 4 hours).
   d. Administer broad-spectrum antibiotics based on suspected infection site and local guidelines.

3. **Monitoring:**
   a. Continuous cardiac monitoring.
   b. Frequent vital signs assessment (every 15 minutes initially, then every hour).
   c. Urine output >0.5 mL/kg/hour.
   d. Serum lactate levels every hour until normalization.
   e. Art

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
prompt_engineered_answers["Q2"] = {
    "profile": prompt_profiles[1]["name"],
    "answer": prompt_engineered_response(questions["Q2"], prompt_profiles[1])
}
print(prompt_engineered_answers["Q2"]["answer"])


Symptoms of Appendicitis:

1. Abdominal pain: The most common symptom is a sharp pain that starts around the navel and migrates to the lower right side of the abdomen. The pain may be constant or come and go, and it may worsen with movement or pressure on the abdomen.
2. Loss of appetite: People with appendicitis often lose their appetite and feel nauseous or vomit.
3. Fever: A fever is present in many cases of appendicitis, with temperatures ranging from 100.4°F (38°C) to 103°F (39.4°C).
4. Rebound tenderness: When pressure is applied to the abdomen and then released, there may be pain or tenderness in the lower right quadrant of the abdomen.
5. Guarding: People with appendicitis may involuntarily contract their abdominal muscles to protect the area from pain.
6. Constipation or diarrhea: Some people experience constipation, while others have diarrhea.
7. Inability to pass gas: A blocked appendix can prevent the passage of gas, leading to bloating and discomfort.

Appendicitis cannot 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
prompt_engineered_answers["Q3"] = {
    "profile": prompt_profiles[2]["name"],
    "answer": prompt_engineered_response(questions["Q3"], prompt_profiles[2])
}
print(prompt_engineered_answers["Q3"]["answer"])


I'm here to help answer any questions you have about health-related concerns, including hair loss. However, I cannot provide definitive diagnoses or recommend specific treatments without consulting a healthcare professional. That being said, I can share some common causes of sudden patchy hair loss and suggest some general approaches that may be helpful.

One possible cause of patchy hair loss is a condition called alopecia areata. This is an autoimmune disease where the body's immune system attacks the hair follicles, leading to hair loss in small, round patches on the scalp or other areas of the body. Other potential causes include stress, nutritional deficiencies, certain medications, and underlying medical conditions such as thyroid disorders.

To address patchy hair loss, it's important to first consult with a healthcare professional for an accurate diagnosis. They may recommend various treatments based on the underlying cause, such as topical corticosteroids or other medications,

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
prompt_engineered_answers["Q4"] = {
    "profile": prompt_profiles[3]["name"],
    "answer": prompt_engineered_response(questions["Q4"], prompt_profiles[3])
}
print(prompt_engineered_answers["Q4"]["answer"])


I'm an AI language model and not a doctor, but I can suggest some general recommendations based on common practices in emergency medicine for someone with a suspected traumatic brain injury (TBI). Please note that this information is intended to be educational and should not replace professional medical advice.

1. Prioritize airway, breathing, and circulation: Ensure the person's airway is open, they are breathing adequately, and their circulation is stable. This may involve performing CPR if necessary or using a cervical collar and backboard to immobilize the spine during transport.

2. Monitor vital signs: Keep track of the person's heart rate, blood pressure, respiratory rate, and oxygen saturation levels.

3. Immobilize the head: Use a cervical collar and securely place the person on a flat, firm surface with their head stabilized to prevent further injury or movement.

4. Assess level of consciousness: Use tools like the Glasgow Coma Scale (GCS) to assess the person's level of co

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [15]:
prompt_engineered_answers["Q5"] = {
    "profile": prompt_profiles[4]["name"],
    "answer": prompt_engineered_response(questions["Q5"], prompt_profiles[4])
}
print(prompt_engineered_answers["Q5"]["answer"])

prompt_engineering_summary = pd.DataFrame([
    {"question": q, "profile": v["profile"], "answer_preview": v["answer"][:350]}
    for q, v in prompt_engineered_answers.items()
])
display(prompt_engineering_summary)
print("Observation: prompt engineering improves structure and safety language, but answers may still contain unsupported details because no manual context has been retrieved yet.")



Title: Leg Fracture During Hiking: Necessary Precautions, Treatment Steps, Contraindications, and Follow-Up Considerations

I. Key Findings:
1. Swelling, bruising, deformity, or inability to bear weight on the affected leg
2. Pain localized to the fracture site
3. Possible open wound (compound fracture)
4. Potential for associated injuries such as soft tissue damage or nerve injury
5. Presence of bone protruding through the skin (in severe cases)

II. Necessary Precautions:
1. Maintain immobilization of the affected leg to prevent further injury and promote healing
2. Avoid weight-bearing activities on the affected leg
3. Provide adequate pain management
4. Monitor for signs of infection, such as increased pain, redness, swelling, or drainage from the wound
5. Ensure proper hydration and nutrition

III. Treatment Steps:
1. Assess the severity of the fracture and determine if it requires emergency intervention (open fractures, severe pain, or inability to bear weight)
2. Immobilize the 

,question,profile,answer_preview
0,Q1,concise_clinical_protocol,"Title: Septic Shock Management Protocol in Critical Care Unit\n\n1. **Recognition:** Suspect sepsis if the patient has an infection, plus two or more of the following: temperat..."
1,Q2,differential_and_treatment,Symptoms of Appendicitis:\n\n1. Abdominal pain: The most common symptom is a sharp pain that starts around the navel and migrates to the lower right side of the abdomen. The pa...
2,Q3,patient_safe_language,"I'm here to help answer any questions you have about health-related concerns, including hair loss. However, I cannot provide definitive diagnoses or recommend specific treatmen..."
3,Q4,trauma_first_aid,"I'm an AI language model and not a doctor, but I can suggest some general recommendations based on common practices in emergency medicine for someone with a suspected traumatic..."
4,Q5,evidence_checklist,"Title: Leg Fracture During Hiking: Necessary Precautions, Treatment Steps, Contraindications, and Follow-Up Considerations\n\nI. Key Findings:\n1. Swelling, bruising, deformity..."


Observation: prompt engineering improves structure and safety language, but answers may still contain unsupported details because no manual context has been retrieved yet.


### Prompt Engineering Observations

| Profile                    | Parameters           | Strength                        | Weakness                | Best Use        |
| -------------------------- | -------------------- | ------------------------------- | ----------------------- | --------------- |
| concise_clinical_protocol  | temp 0.0, top_p 0.90 | Structured, stable              | Less explanatory        | Sepsis protocol |
| differential_and_treatment | temp 0.2             | Good for diagnosis + management | More verbose            | Appendicitis    |
| patient_safe_language      | temp 0.3             | Patient-friendly                | Less clinical detail    | Alopecia        |
| trauma_first_aid           | temp 0.1             | Good emergency framing          | May omit long-term care | Brain injury    |
| evidence_checklist         | temp 0.0             | Good checklist                  | Can be rigid            | Fracture        |

This table summarizes the five prompt-engineering and parameter-tuning profiles tested in the notebook. Each profile was designed for a different medical question type. Lower temperature values were used for clinical accuracy, consistency, and safety, while slightly higher temperature values were used when more patient-friendly explanation was required. The comparison shows that prompt design affects structure, detail level, readability, and clinical usefulness of the LLM response.

Observation:
Prompt engineering improved structure, safety framing, and readability. However, it did not solve the main limitation of base LLM answers: the answers were still not guaranteed to be grounded in the medical manual. This motivates the RAG approach.

## Data Preparation for RAG

### Loading the Data

In [16]:
# Load the Merck medical manual PDF. Keep the PDF in the same folder as this notebook.
manual_pdf_path = "medical_diagnosis_manual.pdf"

pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()
print(f"Loaded {len(manual):,} pages from {manual_pdf_path}")


Loaded 4,114 pages from medical_diagnosis_manual.pdf


### Data Overview

#### Checking the first 5 pages

In [17]:
# Inspect the first five pages. The preview is truncated for readability.
for i, page in enumerate(manual[:5], start=1):
    print(f"\n--- Page {i} ---")
    print(page.page_content[:1200])



--- Page 1 ---
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 2 ---
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 3 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

#### Checking the number of pages

In [18]:
print(f"Number of pages in the manual: {len(manual):,}")


Number of pages in the manual: 4,114


### Data Chunking

In [19]:
# Split pages into overlapping chunks so each retrieved passage has enough clinical context.
# The overlap helps preserve treatment steps that span paragraph boundaries.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

document_chunks = pdf_loader.load_and_split(text_splitter)
print(f"Created {len(document_chunks):,} chunks")
print(document_chunks[0].page_content[:800])


Created 7,203 chunks
akhtar.naheed@gmail.com
HF24TEQP91
This file is meant for personal use by akhtar.naheed@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


### Embedding

In [20]:
# Load the embedding model used to convert text chunks and queries into comparable vectors.
# If you previously saw a tf_keras / TensorFlow import error, restart the notebook kernel
# and run from the imports cell again so USE_TF=0 is applied before transformers loads.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Dimension of the embedding vector:", len(embedding_1))
print("Same dimension for adjacent chunks:", len(embedding_1) == len(embedding_2))


Dimension of the embedding vector: 384
Same dimension for adjacent chunks: True


### Vector Database

In [21]:
# Build and persist the Chroma vector database.
out_dir = "medical_db"
os.makedirs(out_dir, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_model,
    persist_directory=out_dir
)

# Recent Chroma versions persist automatically; call persist when available for compatibility.
if hasattr(vectorstore, "persist"):
    vectorstore.persist()

print(f"Vector database created at: {out_dir}")


Vector database created at: medical_db


### Retriever

In [22]:
# Configure the retriever. Similarity search with k=4 balances enough evidence with prompt size.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

sample_query = questions["Q1"]
rel_docs = vectorstore.similarity_search(sample_query, k=4)
for i, doc in enumerate(rel_docs, start=1):
    page = doc.metadata.get("page", "unknown")
    print(f"\n--- Retrieved chunk {i}; page {page} ---")
    print(doc.page_content[:700])



--- Retrieved chunk 1; page 2400 ---
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and p

--- Retrieved chunk 2; page 2400 ---
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffe

### System and User Prompt Template

In [23]:
qna_system_message = """
You are a medical decision-support assistant using a trusted medical manual as context.
Answer only from the supplied context when possible. If the context is incomplete, say what is missing instead of inventing details.
Use clear clinical language, include immediate safety precautions when relevant, and remind that the output supports but does not replace licensed clinical judgment.
""".strip()

qna_user_message_template = """
Context from the medical manual:
{context}

Question:
{question}

Provide a grounded answer with these sections when applicable:
- Direct answer
- Symptoms or causes
- Treatment / management steps
- Urgent escalation criteria
- Source pages used
""".strip()


### Response Function

In [24]:
def _format_docs_for_context(docs):
    newline = chr(10)
    formatted_chunks = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page")
        source = doc.metadata.get("source", "medical manual")
        page_label = page + 1 if isinstance(page, int) else page
        header = f"[Source {i}: {source}, page {page_label}]"
        formatted_chunks.append(header + newline + doc.page_content)
    return (newline * 2).join(formatted_chunks)


def retrieve_documents(user_input, k=4, search_type="similarity", fetch_k=12):
    """Retrieve relevant chunks with either similarity or MMR search."""
    if search_type == "mmr":
        return vectorstore.max_marginal_relevance_search(user_input, k=k, fetch_k=fetch_k)
    return vectorstore.similarity_search(user_input, k=k)


def generate_rag_response(user_input, k=4, search_type="similarity", fetch_k=12,
                          max_tokens=550, temperature=0.0, top_p=0.90, top_k=30,
                          return_context=False):
    """Generate a grounded answer using retrieved medical-manual chunks."""
    relevant_document_chunks = retrieve_documents(
        user_input=user_input,
        k=k,
        search_type=search_type,
        fetch_k=fetch_k
    )
    context_for_query = _format_docs_for_context(relevant_document_chunks)

    user_message = qna_user_message_template.format(
        context=context_for_query,
        question=user_input
    )

    try:
        answer = llm_chat(
            system_message=qna_system_message,
            user_message=user_message,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )
    except Exception as e:
        answer = "Sorry, I encountered the following error:" + chr(10) + str(e)

    if return_context:
        return answer, context_for_query, relevant_document_chunks
    return answer


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [25]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q1"] = generate_rag_response(questions["Q1"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q1"])


Direct answer:
The protocol for managing sepsis in a critical care unit involves the following steps:

Symptoms or causes:
Sepsis is a life-threatening condition caused by a dysregulated response to infection. It can lead to tissue damage, organ failure, and even death. Symptoms include shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea). Septic shock develops in 25 to 40% of patients with significant bacteremia.

Treatment / management steps:
1. Obtain cultures of blood and any other appropriate specimens.
2. Administer empiric antibiotics after obtaining cultures. Early treatment with an appropriate antimicrobial regimen improves survival.
3. Adjust antibiotics according to culture and susceptibility testing results.
4. Surgically drain any abscesses.
5. Remove internal devices suspected of being the source of bacteria.

Urgent escalation criteria:
If there is suspicion of sepsis or septic shock, immediate act

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [26]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q2"] = generate_rag_response(questions["Q2"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q2"])


The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint, or pain caused by passive internal rotation of the flexed thigh (obturator sign). A low-grade fever is also common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.

Appendicitis cannot be cured via medicine alone. The standard treatment for appendicitis is surgical removal of the appendix through open or laparoscopic appendectomy. Treatment delay increases mortality, so a negative appendectomy rate of 15% is cons

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [27]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q3"] = generate_rag_response(questions["Q3"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q3"])


Direct answer:
Sudden patchy hair loss, also known as alopecia areata, is a condition characterized by the sudden appearance of bald spots on the scalp or other hair-bearing areas. The exact cause of alopecia areata is unknown, but it's believed to be an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers.

Symptoms or causes:
The symptoms of alopecia areata include sudden hair loss in patches, which may affect the scalp, beard, or any other hair-bearing area. The condition is usually not associated with any obvious skin or systemic disorders. However, concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for underlying disorders.

Treatment / management steps:
Multiple treatment options exist for alopecia areata, including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psorale

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [28]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q4"] = generate_rag_response(questions["Q4"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q4"])


A person who has sustained a physical injury to brain tissue may require various treatments and management steps depending on the extent and location of the damage. The Merck Manual describes brain injury as causing cerebral dysfunction syndromes such as agnosia, amnesia, aphasia, and apraxia (Source 3).

Direct answer:
The treatment for a person with brain injury includes clinical evaluation, neuropsychologic testing, laboratory tests, and brain imaging to diagnose the cause and determine the extent of damage. Depending on the diagnosis, specific treatments may include occupational therapy, speech therapy, or medications (Source 3). In some cases, surgery might be necessary to remove hematomas or relieve pressure on the brain (Source 1).

Symptoms or causes:
Brain injuries can result from various causes such as trauma, infarcts, tumors, degeneration, toxicmetabolic disorders, vasculopathy, major trauma, or disseminated cancer (Source 3). The symptoms may include impaired perception, m

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [29]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q5"] = generate_rag_response(questions["Q5"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q5"])


Direct answer: A person with a fractured leg should first receive treatment for any life-threatening injuries. The leg should be immobilized using a splint to prevent further injury and decrease pain. Rest, ice, compression, and elevation (RICE) are recommended for soft-tissue injuries. Pain can be managed with opioids. Definitive treatment may involve reduction, which often requires analgesia or sedation. Closed reduction is preferred when possible, while open reduction involves surgical hardware. A cast is typically used for long-term immobilization.

Symptoms or causes: Symptoms of a fractured leg include pain, swelling, bruising, and difficulty bearing weight on the affected limb. The cause is usually an injury from a fall or other trauma.

Treatment / management steps: Initial treatment includes immobilizing the leg with a splint and providing RICE therapy. Pain should be managed with opioids as needed. Definitive treatment, such as reduction, may be required depending on the seve

### Fine-tuning

In [30]:
# Fine-tune chunk/retrieval/LLM behavior using five RAG configurations.
# These profiles vary k, search strategy, temperature, top_p, top_k, and output length.
rag_tuning_profiles = [
    {"name": "focused_similarity", "k": 3, "search_type": "similarity", "fetch_k": 10, "temperature": 0.0, "top_p": 0.85, "top_k": 20, "max_tokens": 500},
    {"name": "broader_similarity", "k": 5, "search_type": "similarity", "fetch_k": 12, "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650},
    {"name": "mmr_diverse_context", "k": 5, "search_type": "mmr", "fetch_k": 20, "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650},
    {"name": "more_explanatory", "k": 4, "search_type": "similarity", "fetch_k": 12, "temperature": 0.2, "top_p": 0.95, "top_k": 40, "max_tokens": 750},
    {"name": "strict_low_variance", "k": 6, "search_type": "mmr", "fetch_k": 24, "temperature": 0.0, "top_p": 0.80, "top_k": 20, "max_tokens": 700},
]

rag_tuning_results = []
for q_id, question in questions.items():
    for profile in rag_tuning_profiles:
        answer = generate_rag_response(
            question,
            k=profile["k"],
            search_type=profile["search_type"],
            fetch_k=profile["fetch_k"],
            max_tokens=profile["max_tokens"],
            temperature=profile["temperature"],
            top_p=profile["top_p"],
            top_k=profile["top_k"],
        )
        rag_tuning_results.append({
            "question": q_id,
            "profile": profile["name"],
            "k": profile["k"],
            "search_type": profile["search_type"],
            "answer_preview": answer[:450]
        })

tuning_df = pd.DataFrame(rag_tuning_results)
display(tuning_df)
print("Observation: lower-temperature RAG answers are more consistent. MMR is useful when a question spans multiple subtopics, while focused similarity is best for single-protocol questions such as sepsis.")


,question,profile,k,search_type,answer_preview
0,Q1,focused_similarity,3,similarity,"Direct answer: The protocol for managing sepsis in a critical care unit involves obtaining cultures of blood and any other appropriate specimens, administering empiric antibiot..."
1,Q1,broader_similarity,5,similarity,Direct answer:\nThe protocol for managing sepsis in a critical care unit involves early recognition and prompt treatment. Suspected cases should undergo cultures of blood and a...
2,Q1,mmr_diverse_context,5,mmr,"Direct answer:\nThe protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, supportive measures such as respiratory and hemodynamic mana..."
3,Q1,more_explanatory,4,similarity,Direct answer:\nThe protocol for managing sepsis in a critical care unit involves the following steps:\n\nSymptoms or causes:\nSepsis is a life-threatening condition caused by ...
4,Q1,strict_low_variance,6,mmr,"Direct answer: The protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, general supportive measures, and identification and treatment..."
5,Q2,focused_similarity,3,similarity,"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right l..."
6,Q2,broader_similarity,5,similarity,"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right l..."
7,Q2,mmr_diverse_context,5,mmr,"Answer:\nThe common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting ..."
8,Q2,more_explanatory,4,similarity,"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right l..."
9,Q2,strict_low_variance,6,mmr,"Answer:\nThe common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting ..."


Observation: lower-temperature RAG answers are more consistent. MMR is useful when a question spans multiple subtopics, while focused similarity is best for single-protocol questions such as sepsis.


### RAG tuning analysis

| RAG Profile         | Retrieval Method |  k | Observation                                                 |
| ------------------- | ---------------- | -: | ----------------------------------------------------------- |
| focused_similarity  | similarity       |  3 | Best for narrow questions like sepsis                       |
| broader_similarity  | similarity       |  5 | More complete but may include extra context                 |
| mmr_diverse_context | MMR              |  5 | Best for multi-part questions                               |
| more_explanatory    | similarity       |  4 | More readable but slightly more risk of unsupported wording |
| strict_low_variance | MMR              |  6 | Most stable, but can be longer                              |


Best overall configuration:
`mmr_diverse_context` with k=5 and temperature=0.0 gave the best balance of relevance, coverage, and groundedness for multi-part medical questions. For narrow protocol questions, focused similarity with k=3 was faster and sufficiently accurate.

## Output Evaluation

We now use an LLM-as-a-judge method to evaluate the RAG system on two dimensions: groundedness and relevance.

Since the same Mistral model is used for evaluation, the scores should be treated as an initial automated quality check, not as a final clinical validation.

In [31]:
groundedness_rater_system_message = """
You are evaluating whether a medical assistant answer is grounded in the supplied context.
Score Groundedness from 1 to 5:
1 = answer is mostly unsupported or contradicts the context
3 = answer is partially supported but contains gaps or unsupported claims
5 = answer is fully supported by the context with no material unsupported claims
Return: Groundedness Score, Evidence, Unsupported Claims, Improvement Needed.
""".strip()


In [32]:
relevance_rater_system_message = """
You are evaluating whether a medical assistant answer directly addresses the user's medical question.
Score Relevance from 1 to 5:
1 = answer does not address the question
3 = answer addresses part of the question but misses important requested details
5 = answer directly and completely answers all parts of the question
Return: Relevance Score, Missing Details, Strengths, Improvement Needed.
""".strip()


In [33]:
user_message_template = """
### Question
{question}

### Retrieved Context
{context}

### Answer
{answer}
""".strip()


In [34]:
def generate_ground_relevance_response(user_input, k=4, search_type="similarity", fetch_k=12,
                                       max_tokens=500, temperature=0.0, top_p=0.90, top_k=30):
    """Generate a RAG answer and evaluate it for groundedness and relevance."""
    answer, context_for_query, _ = generate_rag_response(
        user_input,
        k=k,
        search_type=search_type,
        fetch_k=fetch_k,
        max_tokens=650,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        return_context=True
    )

    evaluator_user_message = user_message_template.format(
        context=context_for_query,
        question=user_input,
        answer=answer
    )

    groundedness_response = llm_chat(
        system_message=groundedness_rater_system_message,
        user_message=evaluator_user_message,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=top_p,
        top_k=top_k,
    )

    relevance_response = llm_chat(
        system_message=relevance_rater_system_message,
        user_message=evaluator_user_message,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=top_p,
        top_k=top_k,
    )

    return {
        "answer": answer,
        "groundedness_evaluation": groundedness_response,
        "relevance_evaluation": relevance_response,
    }


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [35]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q1"] = generate_ground_relevance_response(questions["Q1"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:\n", evaluation_results["Q1"]["answer"], end="\n\n")
print("GROUNDEDNESS EVALUATION:\n", evaluation_results["Q1"]["groundedness_evaluation"], end="\n\n")
print("RELEVANCE EVALUATION:\n", evaluation_results["Q1"]["relevance_evaluation"])


RAG ANSWER:
 Direct answer:
The protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, supportive measures such as respiratory and hemodynamic management, and identification and treatment of the source of infection. (Source 1, page 2444; Source 3, pages 2996-2997)

Symptoms or causes:
Sepsis is a life-threatening condition caused by an inflammatory response to an infection. It can manifest with non-specific clinical signs such as fever, tachycardia, and altered mental status. Common causative sites of sepsis include the lungs, urinary, biliary, and GI tracts. (Source 3, page 2996)

Treatment / management steps:
1. Rapid empiric antibiotic therapy is recommended based on the suspected site of infection and the patient's clinical status. Antibiotics may be adjusted according to sensitivities and the identified organism. (Source 3, pages 2996-2997)
2. General supportive measures include respiratory and hemodynamic management. (Source 1, page 2444; 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [36]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q2"] = generate_ground_relevance_response(questions["Q2"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:\n", evaluation_results["Q2"]["answer"], end="\n\n")
print("GROUNDEDNESS EVALUATION:\n", evaluation_results["Q2"]["groundedness_evaluation"], end="\n\n")
print("RELEVANCE EVALUATION:\n", evaluation_results["Q2"]["relevance_evaluation"])


RAG ANSWER:
 Answer:
The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting to the right lower quadrant. The pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include Rovsing sign, psoas sign, or obturator sign. A low-grade fever is also common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Pain may not be localized, particularly in infants and children. Tenderness may be diffuse or absent, and bowel movements are usually less frequent or absent if diarrhea is a sign, a retrocecal appendix should be suspected.

Appendicitis cannot be cured via medicine alone as it requires surgical removal of the appendix to prevent com

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [37]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q3"] = generate_ground_relevance_response(questions["Q3"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:\n", evaluation_results["Q3"]["answer"], end="\n\n")
print("GROUNDEDNESS EVALUATION:\n", evaluation_results["Q3"]["groundedness_evaluation"], end="\n\n")
print("RELEVANCE EVALUATION:\n", evaluation_results["Q3"]["relevance_evaluation"])


RAG ANSWER:
 The symptom described, sudden patchy hair loss, is most likely a presentation of Alopecia Areata. This condition is characterized by circular patches of baldness on the scalp, with short broken hairs at the margins that resemble exclamation points (Merck Manual, 2019, p. 860).

Possible causes for Alopecia Areata include autoimmune disorders and genetic susceptibility triggered by unclear environmental factors (Merck Manual, 2019, p. 859). Stressors or systemic illnesses may also contribute to the onset of this condition (Merck Manual, 2019, p. 857).

The primary treatment for Alopecia Areata is corticosteroids, which can be administered topically or orally. Triamcinolone acetonide suspension and potent topical corticosteroids like betamethasone are effective options (Merck Manual, 2019, p. 860). Topical anthralin and minoxidil may also be used in combination with corticosteroids for better results (Merck Manual, 2019, p. 860).

In cases of extensive involvement or chronic

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [38]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q4"] = generate_ground_relevance_response(questions["Q4"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:\n", evaluation_results["Q4"]["answer"], end="\n\n")
print("GROUNDEDNESS EVALUATION:\n", evaluation_results["Q4"]["groundedness_evaluation"], end="\n\n")
print("RELEVANCE EVALUATION:\n", evaluation_results["Q4"]["relevance_evaluation"])


RAG ANSWER:
 A person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, is diagnosed with traumatic brain injury (TBI). The treatment for TBI includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. In severe cases, surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas (Source 2, page 3405).

In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are essential. Subsequently, many patients require rehabilitation to regain lost functions (Source 3, page 3638).

For patients with severe cognitive dysfunction, extensive cognitive therapy is required, which is often begun immediately after the injury and continued for months or years. Spinal cord injuries also require 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [39]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q5"] = generate_ground_relevance_response(questions["Q5"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:\n", evaluation_results["Q5"]["answer"], end="\n\n")
print("GROUNDEDNESS EVALUATION:\n", evaluation_results["Q5"]["groundedness_evaluation"], end="\n\n")
print("RELEVANCE EVALUATION:\n", evaluation_results["Q5"]["relevance_evaluation"])


RAG ANSWER:
 Direct answer: A person who has fractured their leg during a hiking trip should first be assessed for any life-threatening injuries or signs of shock. If there are no such symptoms, the leg should be immobilized using a splint to prevent further injury and decrease pain. The RICE (rest, ice, compression, elevation) protocol can also be applied to help minimize swelling and promote healing. Pain is typically treated with opioids. Definitive treatment for the fracture may involve reduction through closed or open methods, depending on the severity of the injury. Surgical intervention may be necessary for certain types of fractures or those that do not respond to conservative treatment.

Symptoms or causes: Symptoms of a fractured leg include pain, swelling, bruising, deformity, and difficulty bearing weight on the affected limb. The cause is usually an injury from a fall or twisting motion.

Treatment / management steps: Initial treatment includes immobilization with a splint

### Evaluation summary

| Question | Groundedness Score | Relevance Score | Observation |
|---|---:|---:|---|
| Q1 | 5 | 5 | Answer was well grounded in retrieved context and directly addressed sepsis management. |
| Q2 | 5 | 5 | Answer covered symptoms, medicine limitations, and surgical management clearly. |
| Q3 | 5 | 5 | Answer was relevant and grounded, though treatment recommendations should still be verified clinically. |
| Q4 | 5 | 5 | Answer addressed traumatic brain injury precautions and management using retrieved context. |
| Q5 | 5 | 5 | Answer was relevant and grounded; more details on fracture types and surgical options could further improve it. |

Although the LLM evaluator scored the answers highly, this evaluation should be treated as a supportive quality check rather than a substitute for expert clinical review. In production, human clinician feedback and additional automated checks should also be used.


## Actionable Insights and Business Recommendations

### Key Takeaways

- A base LLM can provide broad medical information, but it may omit source-specific details or introduce unsupported statements. This is risky in healthcare decision support.
- Prompt engineering improves answer structure, triage language, and safety framing, but prompt design alone does not guarantee medical groundedness.
- RAG improves trustworthiness by retrieving relevant Merck Manual passages before generation, allowing the assistant to anchor answers in a known reference source.
- The best retrieval configuration depends on the task: focused similarity search works well for narrow protocol questions, while MMR helps when questions contain several subtopics such as symptoms, causes, precautions, treatment, and recovery.
- LLM-as-judge evaluation using groundedness and relevance prompts creates a repeatable quality-control layer before the assistant is exposed to clinical users.

### Recommendations

- Deploy the assistant as a clinician-facing support tool, not as an autonomous diagnostic system. The interface should clearly state that final diagnosis and treatment decisions remain with licensed medical professionals.
- Use RAG answers with source-page references so clinicians can quickly verify the original manual text.
- Maintain a curated medical knowledge base and refresh it on a defined review cycle as medical guidelines change.
- Add guardrails for emergency symptoms, uncertainty, and out-of-scope questions. High-risk cases should direct users to urgent evaluation rather than providing overly confident instructions.
- Track groundedness, relevance, retrieval quality, and clinician feedback in production. These metrics should guide chunk-size, retriever, and prompt updates.

### Expected Business Benefits

- Faster access to reliable medical reference information.
- More standardized responses across care teams.
- Reduced time spent manually searching long medical manuals.
- Better explainability through retrieved evidence.
- Improved governance because model outputs can be evaluated against source documents.


<font size=6 color='blue'>Power Ahead</font>
___